# Projeto Aplicado — Sprint 1 · Base Olist
**Segmentação de Clientes por Valor (RFM e Clusterização) para Otimização de Marketing e Retenção em E-commerce de Moda**

Aluno: Fábio Roberto Gomes Filho · Orientador: Marcos Prochnow · Pós-graduação em Ciência de Dados com IA (XP Educação)

| Sprint 1 | 02/11 – 08/11/2026 |
|---|---|
| R01 | Configurar ambiente Python, repositório GitHub e quadro Trello |
| R02 | Coletar e integrar as tabelas Olist |
| R03 | Limpar e tratar a base Olist |
| R04 | Análise exploratória (EDA) |

> A preparação da base da LLVE (R05) fica em um notebook separado (`PA_Sprint1_LLVE`), para que nenhum dado pessoal apareça neste notebook (LGPD).

---
## R01 · Configuração do ambiente
**Critério de aceite:** ambiente funcional (Drive montado, bibliotecas carregadas, arquivos da Olist localizados) e backlog publicado no Trello.

### 1.1 Montagem do Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### 1.2 Parâmetros do projeto
Todos os caminhos e parâmetros ficam centralizados nesta célula (requisito **RT3 – Reaplicabilidade**).

In [2]:
from pathlib import Path

# Caminhos no Google Drive
BASE_DIR    = Path('/content/drive/MyDrive/PROJETO APLICADO CD/PROJETO APLICADO')
DATASET_DIR = BASE_DIR / 'DATASET'        # CSVs originais da Olist (somente leitura)
OUTPUT_DIR  = BASE_DIR / 'PROCESSADOS'    # bases tratadas geradas pelos notebooks
FIG_DIR     = BASE_DIR / 'FIGURAS'        # gráficos exportados para o relatório

# Reprodutibilidade (RT1)
SEED = 42

for pasta in (OUTPUT_DIR, FIG_DIR):
    pasta.mkdir(parents=True, exist_ok=True)

print('Base   :', BASE_DIR.exists(), BASE_DIR)
print('Dataset:', DATASET_DIR.exists(), DATASET_DIR)

Base   : True /content/drive/MyDrive/PROJETO APLICADO CD/PROJETO APLICADO
Dataset: True /content/drive/MyDrive/PROJETO APLICADO CD/PROJETO APLICADO/DATASET


### 1.3 Bibliotecas e versões

In [3]:
import sys, platform, random
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn, scipy

random.seed(SEED)
np.random.seed(SEED)

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
sns.set_theme(style='whitegrid', context='notebook')

versoes = pd.DataFrame({
    'componente': ['Python', 'pandas', 'numpy', 'matplotlib', 'seaborn', 'scikit-learn', 'scipy'],
    'versão': [platform.python_version(), pd.__version__, np.__version__, matplotlib.__version__,
               sns.__version__, sklearn.__version__, scipy.__version__],
})
versoes

,componente,versão
0,Python,3.13.18
1,pandas,2.2.3
2,numpy,2.4.6
3,matplotlib,3.10.0
4,seaborn,0.13.2
5,scikit-learn,1.6.1
6,scipy,1.16.3


### 1.4 Inventário dos arquivos da Olist
Confere se os 9 arquivos esperados estão na pasta e registra tamanho, linhas e colunas de cada um.

In [4]:
ARQUIVOS_OLIST = {
    'orders':        'olist_orders_dataset.csv',
    'order_items':   'olist_order_items_dataset.csv',
    'payments':      'olist_order_payments_dataset.csv',
    'customers':     'olist_customers_dataset.csv',
    'reviews':       'olist_order_reviews_dataset.csv',
    'products':      'olist_products_dataset.csv',
    'sellers':       'olist_sellers_dataset.csv',
    'geolocation':   'olist_geolocation_dataset.csv',
    'translation':   'product_category_name_translation.csv',
}
USADAS_NO_PROJETO = {'orders', 'order_items', 'payments', 'customers'}   # escopo (Quadro 4)

linhas = []
for nome, arq in ARQUIVOS_OLIST.items():
    caminho = DATASET_DIR / arq
    if caminho.exists():
        df_tmp = pd.read_csv(caminho)
        linhas.append({'tabela': nome, 'arquivo': arq,
                       'tamanho_MB': caminho.stat().st_size / 1e6,
                       'linhas': len(df_tmp), 'colunas': df_tmp.shape[1],
                       'usada_no_projeto': nome in USADAS_NO_PROJETO})
    else:
        linhas.append({'tabela': nome, 'arquivo': arq, 'tamanho_MB': None,
                       'linhas': None, 'colunas': None, 'usada_no_projeto': nome in USADAS_NO_PROJETO})

inventario = pd.DataFrame(linhas).astype({'linhas': 'Int64', 'colunas': 'Int64'})
faltando = inventario[inventario['linhas'].isna()]['arquivo'].tolist()
print('Arquivos encontrados:', len(inventario) - len(faltando), 'de', len(inventario))
if faltando:
    print('⚠️ Faltando:', faltando)
inventario

Arquivos encontrados: 9 de 9


,tabela,arquivo,tamanho_MB,linhas,colunas,usada_no_projeto
0,orders,olist_orders_dataset.csv,17.65,99441,8,True
1,order_items,olist_order_items_dataset.csv,15.44,112650,7,True
2,payments,olist_order_payments_dataset.csv,5.78,103886,5,True
3,customers,olist_customers_dataset.csv,9.03,99441,5,True
4,reviews,olist_order_reviews_dataset.csv,14.45,99224,7,False
5,products,olist_products_dataset.csv,2.38,32951,9,False
6,sellers,olist_sellers_dataset.csv,0.17,3095,4,False
7,geolocation,olist_geolocation_dataset.csv,61.27,1000163,5,False
8,translation,product_category_name_translation.csv,0.00,71,2,False


### 1.5 Checklist do R01
- [x] Google Drive montado e pastas `PROCESSADOS` e `FIGURAS` criadas
- [x] Bibliotecas carregadas e versões registradas (RT1)
- [x] Arquivos da Olist localizados e inventariados
- [x] Repositório GitHub criado e este notebook salvo nele (*Arquivo → Salvar uma cópia no GitHub*)
- [x] Quadro Trello com o backlog R01–R16 publicado

---
## R02 · Coleta e integração das tabelas Olist
**Critério de aceite:** base única com **uma linha por pedido entregue**, contendo valor total e `customer_unique_id`.

Fluxo:
1. Carregar as 4 tabelas do escopo (orders, order_items, payments, customers)
2. Validar as chaves e a cardinalidade das relações
3. Agregar itens e pagamentos por pedido
4. Integrar tudo em uma base por pedido e filtrar os pedidos `delivered`
5. Conferir a integridade e salvar em `PROCESSADOS`

### 2.1 Carga das tabelas

In [5]:
DATAS_ORDERS = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
                'order_delivered_customer_date', 'order_estimated_delivery_date']

orders      = pd.read_csv(DATASET_DIR / ARQUIVOS_OLIST['orders'], parse_dates=DATAS_ORDERS)
order_items = pd.read_csv(DATASET_DIR / ARQUIVOS_OLIST['order_items'], parse_dates=['shipping_limit_date'])
payments    = pd.read_csv(DATASET_DIR / ARQUIVOS_OLIST['payments'])
customers   = pd.read_csv(DATASET_DIR / ARQUIVOS_OLIST['customers'], dtype={'customer_zip_code_prefix': str})

for nome, df in {'orders': orders, 'order_items': order_items, 'payments': payments, 'customers': customers}.items():
    print(f'{nome:<12} {df.shape[0]:>8,} linhas × {df.shape[1]} colunas')

orders         99,441 linhas × 8 colunas
order_items   112,650 linhas × 7 colunas
payments      103,886 linhas × 5 colunas
customers      99,441 linhas × 5 colunas


### 2.2 Validação das chaves e da cardinalidade
Ponto crítico do projeto: na Olist, o `customer_id` é gerado **a cada pedido**. O identificador real do cliente é o `customer_unique_id`. Usar o `customer_id` faria todos os clientes parecerem compradores de uma única vez.

In [6]:
checagens = pd.DataFrame([
    {'verificação': 'order_id único em orders',            'resultado': orders['order_id'].is_unique},
    {'verificação': 'customer_id único em customers',      'resultado': customers['customer_id'].is_unique},
    {'verificação': 'todo pedido tem cliente cadastrado',  'resultado': orders['customer_id'].isin(customers['customer_id']).all()},
    {'verificação': 'relação customer_id → pedido é 1:1',  'resultado': orders['customer_id'].is_unique},
])
display(checagens)

card = pd.DataFrame({
    'medida': ['pedidos (order_id)', 'customer_id distintos', 'clientes reais (customer_unique_id)',
               'itens por pedido (máx.)', 'registros de pagamento por pedido (máx.)'],
    'valor': [orders['order_id'].nunique(), customers['customer_id'].nunique(),
             customers['customer_unique_id'].nunique(),
                   order_items.groupby('order_id').size().max(),
                   payments.groupby('order_id').size().max()],
})
card

,verificação,resultado
0,order_id único em orders,True
1,customer_id único em customers,True
2,todo pedido tem cliente cadastrado,True
3,relação customer_id → pedido é 1:1,True


,medida,valor
0,pedidos (order_id),99441
1,customer_id distintos,99441
2,clientes reais (customer_unique_id),96096
3,itens por pedido (máx.),21
4,registros de pagamento por pedido (máx.),29


### 2.3 Agregação de itens e pagamentos por pedido
Um pedido pode ter vários itens e vários registros de pagamento (ex.: cartão + voucher). Os dois são consolidados para **uma linha por pedido** antes da integração, evitando duplicar valores no merge.

In [7]:
itens_pedido = (order_items
    .groupby('order_id')
    .agg(qtd_itens=('order_item_id', 'count'),
         valor_produtos=('price', 'sum'),
         valor_frete=('freight_value', 'sum'))
    .reset_index())

# forma de pagamento principal = a de maior valor no pedido
pag_principal = (payments.sort_values('payment_value', ascending=False)
                 .drop_duplicates('order_id')[['order_id', 'payment_type']]
                 .rename(columns={'payment_type': 'tipo_pagamento_principal'}))

pagamentos_pedido = (payments
    .groupby('order_id')
    .agg(valor_pago=('payment_value', 'sum'),
         qtd_formas_pagamento=('payment_sequential', 'count'),
         parcelas_max=('payment_installments', 'max'))
    .reset_index()
    .merge(pag_principal, on='order_id', how='left'))

print('Pedidos com itens     :', f'{len(itens_pedido):,}')
print('Pedidos com pagamento :', f'{len(pagamentos_pedido):,}')

Pedidos com itens     : 98,666
Pedidos com pagamento : 99,440


### 2.4 Integração e filtro de pedidos entregues

In [8]:
base = (orders
    .merge(customers[['customer_id', 'customer_unique_id', 'customer_city', 'customer_state']],
           on='customer_id', how='left', validate='one_to_one')
    .merge(itens_pedido, on='order_id', how='left', validate='one_to_one')
    .merge(pagamentos_pedido, on='order_id', how='left', validate='one_to_one'))

print('Distribuição de status (todos os pedidos):')
display(base['order_status'].value_counts().to_frame('pedidos')
        .assign(pct=lambda d: d['pedidos'] / d['pedidos'].sum() * 100))

pedidos = base[base['order_status'] == 'delivered'].copy()

funil = pd.DataFrame({
    'etapa': ['Pedidos na base', 'Pedidos entregues (delivered)'],
    'pedidos': [len(base), len(pedidos)],
    'clientes únicos': [base['customer_unique_id'].nunique(), pedidos['customer_unique_id'].nunique()],
})
funil['% dos pedidos'] = funil['pedidos'] / len(base) * 100
funil

Distribuição de status (todos os pedidos):


,pedidos,pct
order_status,,
delivered,96478,97.02
shipped,1107,1.11
canceled,625,0.63
unavailable,609,0.61
invoiced,314,0.32
processing,301,0.30
created,5,0.01
approved,2,0.00


,etapa,pedidos,clientes únicos,% dos pedidos
0,Pedidos na base,99441,96096,100.00
1,Pedidos entregues (delivered),96478,93358,97.02


### 2.5 Conferência de integridade
- Pedidos entregues sem itens ou sem pagamento
- Diferença entre o valor pago e (produtos + frete). Diferenças pequenas são esperadas por juros de parcelamento e vouchers.

In [9]:
pedidos['dif_pago_vs_itens'] = pedidos['valor_pago'] - (pedidos['valor_produtos'] + pedidos['valor_frete'])

integridade = pd.DataFrame([
    {'verificação': 'entregues sem itens',                      'pedidos': pedidos['qtd_itens'].isna().sum()},
    {'verificação': 'entregues sem pagamento',                  'pedidos': pedidos['valor_pago'].isna().sum()},
    {'verificação': 'entregues sem data de entrega ao cliente', 'pedidos': pedidos['order_delivered_customer_date'].isna().sum()},
    {'verificação': '|valor pago − (produtos + frete)| > R$ 1', 'pedidos': (pedidos['dif_pago_vs_itens'].abs() > 1).sum()},
])
display(integridade)

print('Resumo financeiro dos pedidos entregues (R$):')
pedidos[['valor_pago', 'valor_produtos', 'valor_frete', 'qtd_itens']].describe().T

,verificação,pedidos
0,entregues sem itens,0
1,entregues sem pagamento,1
2,entregues sem data de entrega ao cliente,8
3,|valor pago − (produtos + frete)| > R$ 1,246


Resumo financeiro dos pedidos entregues (R$):


,count,mean,std,min,25%,50%,75%,max
valor_pago,"96,477.00",159.86,218.81,9.59,61.88,105.28,176.33,"13,664.08"
valor_produtos,"96,478.00",137.04,209.05,0.85,45.90,86.57,149.90,"13,440.00"
valor_frete,"96,478.00",22.79,21.56,0.00,13.85,17.17,24.02,"1,794.96"
qtd_itens,"96,478.00",1.14,0.54,1.00,1.00,1.00,1.00,21.00


### 2.6 Exportação da base integrada

In [10]:
COLUNAS_BASE = ['order_id', 'customer_unique_id', 'customer_id', 'order_status',
                'order_purchase_timestamp', 'order_approved_at', 'order_delivered_customer_date',
                'order_estimated_delivery_date', 'customer_city', 'customer_state',
                'qtd_itens', 'valor_produtos', 'valor_frete', 'valor_pago',
                'qtd_formas_pagamento', 'parcelas_max', 'tipo_pagamento_principal']

pedidos = pedidos[COLUNAS_BASE]
arq_saida = OUTPUT_DIR / 'olist_pedidos_integrados.parquet'
pedidos.to_parquet(arq_saida, index=False)

print(f'Salvo: {arq_saida.name}  ·  {pedidos.shape[0]:,} pedidos × {pedidos.shape[1]} colunas  ·  '
      f'{pedidos["customer_unique_id"].nunique():,} clientes únicos')
pedidos.head()

Salvo: olist_pedidos_integrados.parquet  ·  96,478 pedidos × 17 colunas  ·  93,358 clientes únicos


,order_id,customer_unique_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_customer_date,order_estimated_delivery_date,customer_city,customer_state,qtd_itens,valor_produtos,valor_frete,valor_pago,qtd_formas_pagamento,parcelas_max,tipo_pagamento_principal
0,e481f51cbdc54678b7cc49136f2d6af7,7c396fd4830fd04220f754e42b4e5bff,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-10 21:25:13,2017-10-18,sao paulo,SP,1.00,29.99,8.72,38.71,3.00,1.00,voucher
1,53cdb2fc8bc7dce0b6741e2150273451,af07308b275d755c9edb36a90c618231,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-08-07 15:27:45,2018-08-13,barreiras,BA,1.00,118.70,22.76,141.46,1.00,1.00,boleto
2,47770eb9100c2d0c44946d9cf07ec65d,3a653a41f6f9fc3d2a113cf8398680e8,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-17 18:06:29,2018-09-04,vianopolis,GO,1.00,159.90,19.22,179.12,1.00,3.00,credit_card
3,949d5b44dbf5de918fe9c16f97b45f8a,7c142cf63193a1473d2e66489a9ae977,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-12-02 00:28:42,2017-12-15,sao goncalo do amarante,RN,1.00,45.00,27.20,72.20,1.00,1.00,credit_card
4,ad21c59c0840e6cb83a9ceb5573f8159,72632f0f9dd73dfee390c9b22eb56dd6,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-16 18:17:02,2018-02-26,santo andre,SP,1.00,19.90,8.72,28.62,1.00,1.00,credit_card


### 2.7 Checklist do R02
- [x] 4 tabelas do escopo carregadas
- [x] Chaves e cardinalidade validadas (`customer_unique_id` como identificador do cliente)
- [x] Itens e pagamentos agregados por pedido, sem duplicação no merge
- [x] Filtro de pedidos `delivered` aplicado e registrado no funil
- [x] Base integrada salva em `PROCESSADOS/olist_pedidos_integrados.parquet`